In [1]:
# https://light-tree.tistory.com/196

In [2]:
# Add-on Layer: 카테고리 임베딩 레이어(Category Embedding)와 이 둘을 합치는 퓨전 레이어(Linear Layer)만 새로 만듭니다.
# 이렇게 모델 구조를 뜯어고치는 건 나중에 '평점'이나 '페이지 수' 같은 숫자를 꼭 넣어야 할 때 하셔도 늦지 않습니다.
import random
import math
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd

from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader
from torch.utils.data import random_split

from transformers import AutoTokenizer, AutoModel
from transformers import get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType #, AdapterConfig

In [3]:
def set_seed(seed: int = 42):
    random.seed(seed)            # 기본 Python random 고정
    np.random.seed(seed)         # NumPy 랜덤 고정
    torch.manual_seed(seed)      # CPU 연산 랜덤 고정
    torch.cuda.manual_seed(seed) # GPU 모든 디바이스 랜덤 고정
    torch.cuda.manual_seed_all(seed)  # 멀티 GPU일 때

    # 연산 재현성
    torch.backends.cudnn.deterministic = True  # cuDNN 연산을 determinisitc으로 강제
    torch.backends.cudnn.benchmark = False     # CUDA 성능 자동 튜닝 기능 끔 → 완전 재현 가능
    
set_seed(42)

In [4]:
EPOCHS = 20
WARMUP_RATIO = 0.05 # 원래 0.1이었음
# warmup = 학습 초반에 LR을 낮게 시작해서 천천히 올리는 기법 / Transformer, BERT, GPT 등에서 매우 중요
LEARNING_RATE = 1e-3 # v1: 6e-4, v2:4e-4, v3:8e-4, *v4: 1e-3*, v5:2e-3
BATCH_SIZE=128
TEMPERATURE=0.05 # 타우, 높히면 약간의 오차를 더 허용하게 됨
"""
너의 contrastive loss 구조는:

positive 끼리 한 점처럼 뭉치기 / negative에서 멀어지기
+ 온도(temperature)=0.05 같은 작은 값 
→ exp scaling 때문에 gradient 초강함.. 그럼 cluster collapse 되는 경향이 생김
즉, retrieval에는 오히려 불리한 구조.

retrieval은 “카테고리별 한 점에 뭉치기”가 아니라
semantic 분산 유지 + category 분리의 균형을 원하거든.
    
contrastive가 너무 강력해서 embedding이 collapse되므로,
강한 distillation(λ=200+)이 오히려 retrieval 성능을 높였다는 것은 100% 합리적이다.
"""
LAMBDA_VAL = 400.0 # 높히면 semantic, 낮추면 카테고리 분류
# passage일때...
# e5-small => v1:1.0, *v2:500.0*, v3:900.0, *v4:200.0*, v5:50.0, v6:600.0, *v7:350.0*
# e5-base-v2 => v1:400.0, v2:900.0, v3: 100.0
# query일땐 400이 젤 조흔듯?

model_name = f"400_1e-3_0.05"

In [5]:
import wandb

wandb.init(
    project="retrieval-contrastive", 
    # name=f"e5b2-lora-kd-{LAMBDA_VAL}",
    name = model_name,
    config={
        "lr": LEARNING_RATE,
        "temperature": TEMPERATURE,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
    }
)
# wandb.watch(model, log="all")  # gradients, parameters 기록

/home/0uk/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/0uk/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using

In [6]:
model_name = "intfloat/e5-small" 
# model_name = "intfloat/e5-base-v2" 
tokenizer = AutoTokenizer.from_pretrained(model_name)

teacher_model = AutoModel.from_pretrained(model_name)
teacher_model.eval()  # 평가 모드 (Dropout 등 비활성화)
for param in teacher_model.parameters():
    param.requires_grad = False # 절대 학습되지 않도록 잠금

# 선생님과 별개의 새로운 인스턴스를 만들어야 함!
student_base_model = AutoModel.from_pretrained(model_name)

"""
Transformer 내부의 특정 Layer(Wq, Wk, Wv 등)에 "LoRA 모듈"을 주입!
E5 Model (pretrained)
    ├── LayerNorm
    ├── 24× Transformer Layers
    │       ├── Attention (Q,K,V,O)
    │       ├── FFN Layers
    │       └── ... (원래 weight 고정)
    └── Pooler → embedding vector   # e.g., CLS embedding

LoRA Injected:
    ├── W_q = W_q + A_q B_q
    ├── W_k = W_k + A_k B_k
    ├── W_v = W_v + A_v B_v
    ├── FFN = FFN + A_ffn B_ffn
    └── (small learnable matrices only)
"""
lora_config = LoraConfig( # Linear(d → d) -→ Linear(d → d) + LoRA(d → d)
    task_type=TaskType.FEATURE_EXTRACTION,  # 임베딩 fine-tuning
    # LoRA가 분류기와 같은 output head에 적용되는 것이 아니라
    # 모델의 Transformer 블록(encoder)에만 적용되도록 
    r=16,    # LoRA rank
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none"
)
model = get_peft_model(student_base_model, lora_config)
# 원래 weight(W)는 freeze! 훈련 과정에서 업데이트되지 않음.
# LoRA에서 추가된 A, B 행렬만 학습 학습량은 전체의 0.1%~1% 수준으로 줄어듦.
# forward 시에는 LoRA의 low-rank update가 추가됨 ex) outputs = W x + (B(Ax)) * (α / r)

device = "cuda" if torch.cuda.is_available() else "cpu"
teacher_model.to(device)
model.to(device)

PeftModelForFeatureExtraction(
  (base_model): LoraModel(
    (model): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(30522, 384, padding_idx=0)
        (position_embeddings): Embedding(512, 384)
        (token_type_embeddings): Embedding(2, 384)
        (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSdpaSelfAttention(
                (query): lora.Linear(
                  (base_layer): Linear(in_features=384, out_features=384, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.1, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=384, out_features=16, bias=False)
                  )
            

In [7]:
# book_path = './data/e5_book_meta.parquet'
book_path = './data/book_for_emb.parquet'
books = pd.read_parquet(book_path)

In [8]:
def build_text(row): # 입력 텍스트 생성 (타이틀 + 설명 + 저자 등 결합)
    parts = [
        f"Title: {row['title']} |",
        # f"Category: {row['category']} |",
        f"Description: {row['description']}"
    ]
    return " ".join( # 리스트의 문자열들을 공백으로 연결할건데.....
        [p for p in parts if isinstance(p, str)] # NaN이나 None이 있으면 제외함
    ) # 최종적으로 하나의 문장 형태로 반환한다고 함!! "Title: ... Category: ... Description: ..."

books["text"] = books.apply(build_text, axis=1) # 새 컬럼 text에 대해서.... 문장 만듦

# 100개 미만인 카테고리는 노이즈로 간주하고 삭제
counts = books['category'].value_counts()
valid_categories = counts[counts > 100].index
books = books[books['category'].isin(valid_categories)]

In [9]:
dataset = Dataset.from_pandas(books)

le = LabelEncoder()
le.fit(dataset['category'])   # 전체 데이터로 학습

def encode_label(x):
    return {"label": le.transform([x["category"]])[0]}

dataset = dataset.map(encode_label)

num_classes = len(le.classes_)

Map:   0%|          | 0/81845 [00:00<?, ? examples/s]

In [10]:
"""
collate_fn은 raw text와 label을 텐서로 묶어 모델이 학습할 수 있는 형태로 만들어줌
DataLoader는 이 함수로 미리 전처리한 batch를 모델에 공급하는 역할을 함

Dataset row(dict)
     ↓ (DataLoader)
batch = [row1, row2, ...] (list)
     ↓ (collate_fn)
텍스트 리스트 + 라벨 리스트
     ↓ (tokenizer)
input_ids, attention_mask (tensor)
     ↓
(inputs, labels)
     ↓
model(**inputs)
"""

# Transformer 모델은 이런 raw 텍스트를 바로 처리 못 하고
# 토크나이저를 거쳐 tensor(batch_input_ids, batch_attention_mask) 형태가 필요함.
def collate_fn(batch): # DataLoader가 batch마다 호출
    # texts = [f"passage: {x['text']}" for x in batch]
    texts = [f"query: {x['text']}" for x in batch]
    labels = torch.tensor([x['label'] for x in batch])  # 라벨을 int 리스트 → torch.tensor 로 변환

    """
    토크나이저:
    텍스트를 token id로 변환 (input_ids), attention_mask 생성,
    batch의 최대 length에 맞춰 패딩, 출력 타입은 PyTorch tensor

    { 'input_ids': tensor([[101,  ... , 102], ...]),
      'attention_mask': tensor([[1,1,1,0,0...], ...) }
    """
    inputs = tokenizer(
      texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
    
    return inputs, labels

In [11]:
total_len = len(dataset)
train_len = int(total_len * 0.8)
valid_len = total_len - train_len

train_dataset, valid_dataset = random_split(dataset, [train_len, valid_len])

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)
valid_loader = DataLoader(
    valid_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)

In [12]:
def compute_retrieval_accuracy(model, dataloader, device, k=10):
    embeddings_list = []
    labels_list = []

    with torch.no_grad():
        for batch_inputs, labels in dataloader:
            batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
            labels = labels.to(device)

            outputs = model(**batch_inputs)
            embeddings = outputs.last_hidden_state.mean(dim=1)
            embeddings = F.normalize(embeddings, p=2, dim=1)

            embeddings_list.append(embeddings)
            labels_list.append(labels)

    all_embeddings = torch.cat(embeddings_list, dim=0)
    all_labels = torch.cat(labels_list, dim=0)
    similarity = torch.matmul(all_embeddings, all_embeddings.T)
    similarity.fill_diagonal_(-1e9)

    # nn_index = similarity.argmax(dim=1) # 가장 유사한 이웃 인덱스
    # nn_labels = all_labels[nn_index] # nearest neighbor label
    # accuracy = (nn_labels == all_labels).float().mean().item()

    topk_vals, topk_idx = similarity.topk(k, dim=1) # top-k neighbor 인덱스
    nn_labels_topk = all_labels[topk_idx]  # (N, k) 
    # correct_ratio = (nn_labels_topk == all_labels.unsqueeze(1)).float().mean(dim=1)
    # accuracy = correct_ratio.mean().item()
    # return accuracy

    # -----------------------------
    # top-1: 가장 가까운 neighbor 정답 여부
    # top-k: k개 안에 정답이 하나라도 있으면
    # precision@k: k개 neighbor 중 정답 비율 평균
    # MRR: 정답이 rank 몇 번째인지에 따른 평균 역수 → rank 1이면 1.0, rank 2이면 0.5
    # -----------------------------
    # Top-1 accuracy
    top1_labels = nn_labels_topk[:, 0]
    top1_acc = (top1_labels == all_labels).float().mean().item()
    
    # Top-k accuracy (at least 1 match)
    topk_match = (nn_labels_topk == all_labels.unsqueeze(1)).any(dim=1).float()
    topk_acc = topk_match.mean().item()

    # Precision@k
    precision_at_k = (nn_labels_topk == all_labels.unsqueeze(1)).float().mean(dim=1).mean().item()
    
    # MRR (Mean Reciprocal Rank)
    ranks = (nn_labels_topk == all_labels.unsqueeze(1)).float() # 정답 label 위치 찾기
    reciprocal_rank = []
    for i in range(ranks.size(0)):
        pos_positions = torch.nonzero(ranks[i]).flatten()
        if len(pos_positions) == 0: # positive 없으면 reciprocal rank = 0
            reciprocal_rank.append(0.0)
        else:
            reciprocal_rank.append(1.0 / (pos_positions[0].item() + 1))
    mrr = sum(reciprocal_rank) / len(reciprocal_rank)

    metrics = {
        "top1_acc": top1_acc,
        "topk_acc": topk_acc,
        "precision@k": precision_at_k,
        "mrr": mrr
    }
    return metrics

In [13]:
# 일단 한번 해보고 개선되면 memory bank 식으로 바꿔보자
# 근데 배치가 크면 굳이 memory bank로 안바꿔도 된다는디?
def hard_negative(embeddings, labels, k=3):
    batch_size = embeddings.size(0)
    similarity = torch.matmul(embeddings, embeddings.T) # 1. 유사도 행렬 계산
    
    # hard_neg_sims = []
    # for i in range(batch_size): # for문 대신 행렬 연산(Vectorization)으로 바꾸면 빨라진대
    #     mask = labels != labels[i]           # 다른 라벨만
    #     sim_row = similarity[i].clone()
    #     sim_row[~mask] = -1e9                # 같은 라벨 제외
    #     topk_sim, _ = sim_row.topk(k)        # 가장 가까운 k개
    #     hard_neg_sims.append(topk_sim)
    # hard_neg_sims = torch.stack(hard_neg_sims, dim=0)  # (batch_size, k)

    # 2. Positive Mask 생성 (같은 라벨인 곳 True)
    # labels: (N) -> (N, 1) == (1, N) 브로드캐스팅
    pos_mask = labels.unsqueeze(1) == labels.unsqueeze(0)

    # 3. 같은 라벨(Positive + 자기자신)은 유사도를 -무한대로 보내서 topk에서 제외
    # clone()을 안 쓰면 원본 similarity가 망가지므로 주의 (필요시)
    sim_for_neg = similarity.clone() 
    sim_for_neg.masked_fill_(pos_mask, -1e9)  # 같은 라벨 제외

    # 4. 각 행(Query)마다 가장 높은 유사도를 가진 k개(Hard Negatives) 추출
    hard_neg_sims, _ = sim_for_neg.topk(k, dim=1) # (N, k)

    return hard_neg_sims

In [14]:
total_steps = len(train_loader) * EPOCHS

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
# AdamW 옵티마이저로 LoRA 파라미터만 학습
# LoRA 덕분에 실제 업데이트되는 파라미터는 전체의 1% 정도

scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=int(total_steps * WARMUP_RATIO),
    num_training_steps=total_steps
)

In [15]:
metrics = compute_retrieval_accuracy(model, valid_loader, device)
wandb.log({
    "epoch": 0,
    "valid/top1_acc": metrics["top1_acc"],
    "valid/top10_acc": metrics["topk_acc"],
    "valid/precision@10": metrics["precision@k"],
    "valid/mrr": metrics["mrr"],
    "lr": LEARNING_RATE,
})
print(f"[Before Training...]")
print(f"Top-1 Accuracy : {metrics['top1_acc']:.4f} | Top-10 Accuracy : {metrics['topk_acc']:.4f}")
print(f"Precision@10   : {metrics['precision@k']:.4f} | MRR             : {metrics['mrr']:.4f}")

for epoch in range(EPOCHS):
    """
    Cosine Decay schedule for lambda:
    λ_t = λ0 * 0.5 * (1 + cos(pi * epoch / total_epochs))
    """
    # lambda_t = LAMBDA_VAL * 0.5 * (1 + math.cos(math.pi * epoch / EPOCHS))
    # lambda_t = LAMBDA_VAL * 0.5 * (1 + math.cos(math.pi * epoch / (2 * EPOCHS)))
    lambda_t = LAMBDA_VAL
    
    model.train()
    total_train_loss = 0
    total_loss_contrastive = 0                    # 디버깅
    total_loss_distill = 0                        # 디버깅

    for batch_inputs, labels in tqdm(train_loader, desc = f"Epoch: {epoch+1}"):
        batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
        labels = labels.to(device)

        # 1. Optimizer 초기화 (가장 먼저!)
        optimizer.zero_grad()

        # 2. Teacher (Original E5) Forward
        with torch.no_grad(): # 원본 모델에게 물어봅니다. "너라면 이거 어디에 배치할래?"
            teacher_outputs = teacher_model(**batch_inputs)
            teacher_embeddings = teacher_outputs.last_hidden_state.mean(dim=1)
            teacher_embeddings = F.normalize(teacher_embeddings, p=2, dim=1)

        # 3. Student (LoRA) Forward
        outputs = model(**batch_inputs)
        embeddings = outputs.last_hidden_state.mean(dim=1) # mean pooling
        # 모든 토큰 벡터를 평균 → 하나의 sentence embedding
        embeddings = F.normalize(embeddings, p=2, dim=1)
        # 벡터를 unit vector로 만들어서 cosine similarity 계산 가능하게 함

        # --- Loss Calculation Start ---
        """
        1. loss_contrastive (카테고리 로스): "같은 카테고리니까 뭉쳐! 다른 카테고리는 저리 가!"
            → 힘의 방향: 공간 왜곡 (Distortion)
        
        2. loss_distill (보존 로스): "야, 그래도 원래 의미(Semantic)가 '우주'랑 'SF'였잖아. 그 근처에 있어."
            → 힘의 방향: 원래 자리 유지 (Preservation)
        
        결과적으로 모델은 원래 의미 공간을 최대한 유지하면서, 카테고리 분류가 가능할 정도로만 살짝 이동하게 됨
        """
        # https://www.blossominkyung.com/deeplearning/contrastive-learning
        # 같은 카테고리는 가까이, 다른 카테고리는 멀리
        similarity = torch.matmul(embeddings, embeddings.T)

        # Mask 생성
        labels_eq = labels.unsqueeze(1) == labels.unsqueeze(0)
        identity_mask = torch.eye(len(labels), device=labels.device).bool() # 자기 자신 제거 mask
        labels_eq = labels_eq & (~identity_mask)

        pos_mask = labels_eq.float()   # (N, N)

        """
        pos_sim: anchor와 positive 샘플들의 유사도 벡터
        → 한 anchor가 여러 개의 positive를 가질 수 있음

        neg_sim: anchor와 negative 샘플들의 유사도 벡터
        → 당연히 여러 negative가 존재
        """
        # 정답이 아닌 곳(0)에는 아주 작은 값(-1e9)을 넣어서 exp 계산 시 0이 되도록 해야 함
        # pos_sim = similarity.masked_fill(~pos_mask.bool(), -1e9)
        pos_sim = similarity * pos_mask # 이거 넣으면... 정답 아닌 곳 => 0 => e^0=1 => 1이 쌓인다는데....
        # 난 그거 더 성능이 좋게 나오던데????? 수학적으론 틀려도.....
        # 노이즈가 섞여있어서 한 점으로 덜 뭉쳐서 뭐 retrieval이 더 좋게 나온다던데..?
        """
        exp(0)=1이 일정한 baseline noise 역할
        positive가 너무 한 점으로 collapse 되는 것을 방지
        embedding space가 overly tight cluster로 수렴하지 않음
        retrieval에서는 적절한 분산이 성능을 올리는 경우가 많음
        즉, “정석적인 contrastive”에서는 틀렸지만
        retrieval task에서는 종종 regularization 역할이 되어 더 잘되는 경우가 많음.
        """
        neg_sim = hard_negative(embeddings, labels) # (N, k) → anchor i의 negative k개

        # loss 확대: 정답(0.8/0.05=16), 오답(0.7/0.05=14) => exp(16) ≈ 8,886,110 vs exp(14) ≈ 1,202,604 7배 이상 차이남
        # => 0.1 차이도 크게 만들어 모델이 pos를 더더더 1에 가깝도록 맞춤
        pos_sim = pos_sim / TEMPERATURE
        neg_sim = neg_sim / TEMPERATURE

        # 1. supervised contrastive sum over positives
        # 모든 positive를 합산해서 ratio 계산 → 카테고리별 embedding을 한 점에 모으는 경향
        loss_contrastive = -torch.log(
            torch.exp(pos_sim).sum(dim=1) / 
            (torch.exp(pos_sim).sum(dim=1) + torch.exp(neg_sim).sum(dim=1))
        ).mean()
        # sum해주는 이유는.. 한 anchor(기준 샘플)당 여러 개의 pos/neg 쌍이 존재할 수 있기 때문

        # 2. distillation loss (MSE)
        # 카테고리 맞추는 것도 좋은데, 원래 위치랑 너무 달라지진 마
        loss_distill = F.mse_loss(embeddings, teacher_embeddings)

        # 3. 최종 Loss 합산
        total_loss = loss_contrastive + (lambda_t * loss_distill)
    
        # total_loss.backward()  # 기울기(Gradient) 계산 완료
        # optimizer.zero_grad()  # 기울기를 0으로 초기화 (삭제) 😱
        # optimizer.step()       # 0이 된 기울기로 가중치 업데이트 (변화 없음)
        optimizer.zero_grad()  # 1. 이전 기울기 청소
        total_loss.backward()  # 2. 현재 기울기 계산
        optimizer.step()       # 3. 업데이트
        
        scheduler.step() # 스케줄러가 step 단위라면 여기, epoch 단위면 loop 밖으로
        
        total_train_loss += total_loss.item()
        total_loss_contrastive += loss_contrastive.item()                   # 디버깅
        total_loss_distill += loss_distill.item()                           # 디버깅
        
    train_loss = total_train_loss / len(train_loader)
    train_loss_contrastive = total_loss_contrastive / len(train_loader)           # 디버깅
    train_loss_distill = total_loss_distill / len(train_loader)                   # 디버깅

    model.eval()
    metrics = compute_retrieval_accuracy(model, valid_loader, device)
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "loss_contrastive": train_loss_contrastive,
        "loss_distill": train_loss_distill,
        "valid/top1_acc": metrics["top1_acc"],
        "valid/top10_acc": metrics["topk_acc"],
        "valid/precision@10": metrics["precision@k"],
        "valid/mrr": metrics["mrr"],
        "lr": scheduler.get_last_lr()[0],
    })
    print(f"[Epoch {epoch + 1}] Train Loss: {train_loss:.4f}")
    print(f"Top-1 Accuracy : {metrics['top1_acc']:.4f} | Top-10 Accuracy : {metrics['topk_acc']:.4f}")
    print(f"Precision@10   : {metrics['precision@k']:.4f} | MRR              : {metrics['mrr']:.4f}")

[Before Training...]
Top-1 Accuracy : 0.4739 | Top-10 Accuracy : 0.3715
Precision@10   : 0.3715 | MRR             : 0.5982


Epoch: 1: 100%|████████████████████████████████████████████████████████████████████████| 512/512 [02:18<00:00,  3.69it/s]


[Epoch 1] Train Loss: 1.1755
Top-1 Accuracy : 0.5242 | Top-10 Accuracy : 0.4406
Precision@10   : 0.4406 | MRR              : 0.6416


Epoch: 2: 100%|████████████████████████████████████████████████████████████████████████| 512/512 [02:19<00:00,  3.67it/s]


[Epoch 2] Train Loss: 1.1323
Top-1 Accuracy : 0.5446 | Top-10 Accuracy : 0.4762
Precision@10   : 0.4762 | MRR              : 0.6563


...

[Epoch 18] Train Loss: 1.0619
Top-1 Accuracy : 0.5818 | Top-10 Accuracy : 0.5411
Precision@10   : 0.5411 | MRR              : 0.6743


Epoch: 19: 100%|███████████████████████████████████████████████████████████████████████| 512/512 [02:19<00:00,  3.67it/s]


[Epoch 19] Train Loss: 1.0550
Top-1 Accuracy : 0.5830 | Top-10 Accuracy : 0.5425
Precision@10   : 0.5425 | MRR              : 0.6747


Epoch: 20: 100%|███████████████████████████████████████████████████████████████████████| 512/512 [02:19<00:00,  3.67it/s]


[Epoch 20] Train Loss: 1.0587
Top-1 Accuracy : 0.5832 | Top-10 Accuracy : 0.5421
Precision@10   : 0.5421 | MRR              : 0.6748


In [16]:
torch.save(model.state_dict(), model_name)

RuntimeError: Parent directory intfloat does not exist.

In [ ]:
ㅋㅋ

In [ ]:
# LR Range Test(Leslie Smith의 Cyclical LR 논문에서 나온 방식)

temperature = 0.05
HARD_NEG_K = 3 
LR_START = 1e-7 # LR 탐색 시작점
GAMMA = 1.05    # LR 증가율 (스텝마다 5%씩 증가)
MAX_STEPS = 200 # 탐색 스텝 수
device = "cuda" if torch.cuda.is_available() else "cpu"

# ⚠️ 주의: 실제 데이터셋과 DataLoader는 정의되어 있다고 가정합니다.
# total_len = len(dataset)
# train_dataset, valid_dataset = random_split(dataset, [train_len, valid_len])
# train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, collate_fn=collate_fn)
# -----------------------------------------------------------------------------------

## 🚀 Learning Rate 탐색을 위한 전체 코드 ##
print(f"--- 🚀 LR Range Test 시작: {LR_START:.8f}에서 시작 (스텝마다 {GAMMA}배 증가) ---")

optimizer = optim.AdamW(model.parameters(), lr=LR_START) # 아주 작은 LR로 시작
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=GAMMA) # LR을 지수적으로 증가

model.train()
logs = []

# train_loader 대신 tqdm을 사용하여 진행 상황을 시각화할 수 있습니다.
for i, (batch_inputs, labels) in tqdm(enumerate(train_loader), total=MAX_STEPS):
    batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}
    labels = labels.to(device)

    # 1. Optimizer 초기화 (가장 먼저!)
    optimizer.zero_grad()

    # 2. Teacher (Original E5) Forward
    with torch.no_grad(): # 원본 모델에게 물어봅니다. "너라면 이거 어디에 배치할래?"
        teacher_outputs = teacher_model(**batch_inputs)
        teacher_embeddings = teacher_outputs.last_hidden_state.mean(dim=1)
        teacher_embeddings = F.normalize(teacher_embeddings, p=2, dim=1)

    # 3. Student (LoRA) Forward
    outputs = model(**batch_inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1)
    embeddings = F.normalize(embeddings, p=2, dim=1)

    similarity = torch.matmul(embeddings, embeddings.T)

    labels_eq = labels.unsqueeze(1) == labels.unsqueeze(0)
    identity_mask = torch.eye(len(labels), device=labels.device).bool() # 자기 자신 제거 mask
    labels_eq = labels_eq & (~identity_mask)

    pos_mask = labels_eq.float()   # (N, N)

    pos_sim = similarity * pos_mask
    neg_sim = hard_negative(embeddings, labels)

    pos_sim = pos_sim / TEMPERATURE
    neg_sim = neg_sim / TEMPERATURE

    loss_contrastive = -torch.log(
        torch.exp(pos_sim).sum(dim=1) / 
        (torch.exp(pos_sim).sum(dim=1) + torch.exp(neg_sim).sum(dim=1))
    ).mean()

    loss_distill = F.mse_loss(embeddings, teacher_embeddings)

    loss = loss_contrastive + (LAMBDA_VAL * loss_distill)

    loss.backward()  # 2. 현재 기울기 계산
    optimizer.step()       # 3. 업데이트
    
    # ⚠️ RuntimeError 방지 팁: Loss 값이 너무 커지면 Gradient가 폭발하여 Nan이 발생할 수 있습니다.
    if torch.isnan(loss) or loss.item() > 100:
        print(f"\nLoss 폭발 (Step {i}): {loss.item():.4f}. 탐색 중단.")
        break
        
    optimizer.step()
    scheduler.step() # LR 키우기

    # 6. 로그 기록
    current_lr = optimizer.param_groups[0]['lr']
    logs.append({'step': i, 'lr': current_lr, 'loss': loss.item()})
    
    # 현재 LR과 Loss 출력
    if i % 10 == 0: # 너무 자주 출력되지 않도록 10 스텝마다 출력
        print(f"Step {i}, LR: {current_lr:.8f}, Loss: {loss.item():.4f}")

    if i >= MAX_STEPS:
        break

print("--- 🔍 LR Range Test 완료 ---")

In [ ]:
import matplotlib.pyplot as plt

plt.plot([l['lr'] for l in logs], [l['loss'] for l in logs])
plt.xscale('log')
plt.xlabel("Learning Rate")
plt.ylabel("Loss")
plt.show()